# HE regression heritability/shared-environment estimators

Fits the estimator table from `notes.md` (six estimators: `h2_Unrel`, `h2_FS`,
`h2_Ped,W25`, `h2_Ped*f`, `b2_FS`, `b2_step`) against `grm_shard_processing.ipynb`'s
already-binned, merged crossproduct output (`~/grm_pheno_cov/{tag}_merged.full.tsv`,
one file per `(phenotype, transform, covariate_set)` combo -- see that notebook's
`run_combo`). Run `grm_shard_processing.ipynb` first; this notebook doesn't
call `grm_shard_tool` itself, it only reads that notebook's output.

Each `*_merged.full.tsv` row is one relatedness bin: `bin_midpoint` (mean
pairwise relatedness `a_ij` in the bin), `full_mean` (mean phenotype
cross-product `\tilde{y}_i\tilde{y}_j` in the bin), `full_n` (pair count),
`jk_se` (block-jackknife standard error of the bin mean, `NBLOCKS=50` blocks
-- see `grm_shard_tool merge`). All regressions here are **bin-level weighted
least squares**, not the pair-level `lm()` the legacy
`GRM-pairs/regress_y/estimator_fits.R` used -- this pipeline never assembles
a per-pair table (it's binned by `grm_shard_tool accumulate` before this
notebook ever sees it), so per-bin jackknife SEs stand in for pair-level
bootstrap. Weighting convention matches `grm_shard_processing.ipynb`'s own
dashed slope-line fit: weight `w = 1/jk_se` passed the way `numpy.polyfit`
expects it (which is algebraically the standard WLS weight `1/jk_se**2` in
the normal equations -- same fit, different API convention), so a bin's
influence on the estimate is inverse-variance-weighted.

No bootstrap CI in this first pass -- WLS's own analytic standard errors
(from the weighted residual variance) are reported instead of re-bootstrapping
over already-binned data. Revisit if that turns out to be too optimistic.

## Setup

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WORK_DIR = os.path.expanduser("~/grm_pheno_cov")   # grm_shard_processing.ipynb's merge output lives here

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)

# Must match whatever CDR_VERSION 01_ancestry_filtering/02_phenotype/
# 03_grm_shards notebooks were actually run with -- this stage reads their output.
CDR_VERSION = "v9"
BUCKET_DIR = f"{WORKSPACE_BUCKET}/{CDR_VERSION}/04_process_shards/he_regression_estimators"
os.makedirs(BUCKET_DIR, exist_ok=True)   # results/plots persisted here, not just local -- survives VM deletion

print("WORK_DIR:", WORK_DIR)
print("BUCKET_DIR:", BUCKET_DIR)

## Discover merged combos

Same data-driven discovery pattern as `grm_shard_processing.ipynb`'s
`combos` cell -- globs for whatever `*_merged.full.tsv` files that notebook
has actually produced, rather than hardcoding a phenotype/transform/covset
list. Filename convention: `{phenotype_name}__{transform}__{covariate_set}_merged.full.tsv`.

In [ ]:
merged_files = sorted(glob.glob(f"{WORK_DIR}/*_merged.full.tsv"))
assert merged_files, f"No *_merged.full.tsv files found under {WORK_DIR} -- run grm_shard_processing.ipynb first"

combos = []
for path in merged_files:
    stem = os.path.basename(path)[: -len("_merged.full.tsv")]
    parts = stem.split("__", 2)
    if len(parts) != 3:
        print(f"  skipping unparseable filename: {path}")
        continue
    phenotype_name, transform, covariate_set = parts
    combos.append((phenotype_name, transform, covariate_set, path))

print(f"{len(combos)} phenotype/transform/covariate_set combos found")
for c in combos[:10]:
    print(" ", c[:3])

## Estimator definitions

Ported from `notes.md`'s table (and cross-checked against the legacy
`GRM-pairs/regress_y/estimator_fits.R`, which implements the same models
at the pair level: `h2_unrel`/`h2_FS` -> no-intercept slope,
`h2_ped_W26` -> quadratic no-intercept (`h2 = beta_a`), `h2_ped_SE_factors`
-> slope + indicator offsets at 0.1/0.2/0.4/0.6 (`h2 = beta_a`)).

`b2_step`'s two ranges (FS: 0.4-0.6, HS: 0.2-0.3) match the legacy `h2_FS`/
`h2_HS` ranges.

In [ ]:
# term names resolved by build_design() below
ESTIMATORS = [
    {"name": "h2_Unrel",   "range": (-np.inf, 0.02), "terms": ["a"],                     "intercept": False},
    {"name": "h2_FS",      "range": (0.4, 0.6),      "terms": ["a"],                     "intercept": False},
    {"name": "h2_Ped_W25", "range": (0.05, 0.7),     "terms": ["a", "a2"],                "intercept": False},
    {"name": "h2_Ped_f",   "range": (0.05, 0.7),     "terms": ["a", "f1", "f2", "f3"],    "intercept": False},
    {"name": "b2_FS",      "range": (0.4, 0.6),      "terms": ["a"],                     "intercept": True},
]
# indicator ranges 0.1<=a<0.2, 0.2<=a<0.4, 0.4<=a<0.6, matching h2_Ped_f's spec
B2_STEP_FS_RANGE = (0.4, 0.6)
B2_STEP_HS_RANGE = (0.2, 0.3)   # matches legacy h2_HS range


def build_design(a, terms):
    cols = []
    for t in terms:
        if t == "a":
            cols.append(a)
        elif t == "a2":
            cols.append(a ** 2)
        elif t == "f1":
            cols.append(((a >= 0.1) & (a < 0.2)).astype(float))
        elif t == "f2":
            cols.append(((a >= 0.2) & (a < 0.4)).astype(float))
        elif t == "f3":
            cols.append(((a >= 0.4) & (a < 0.6)).astype(float))
        else:
            raise ValueError(f"unknown term: {t}")
    return np.column_stack(cols)


def wls_fit(X, y, se):
    # Standard weighted least squares, weight = 1/se**2 -- the same fit
    # numpy.polyfit(..., w=1/se) produces (polyfit's w is applied unsquared to
    # both X and y, which is algebraically identical to solving the normal
    # equations with weight 1/se**2).
    w = 1.0 / np.asarray(se) ** 2
    XtW = X.T * w
    XtWX = XtW @ X
    beta = np.linalg.solve(XtWX, XtW @ y)
    resid = y - X @ beta
    dof = len(y) - X.shape[1]
    if dof <= 0:
        return beta, np.full_like(beta, np.nan), dof
    sigma2 = np.sum(w * resid ** 2) / dof
    cov = sigma2 * np.linalg.inv(XtWX)
    return beta, np.sqrt(np.diag(cov)), dof


def usable_bins(df, lo, hi):
    sub = df[(df["bin_midpoint"] >= lo) & (df["bin_midpoint"] <= hi)]
    return sub[(sub["full_n"] > 0) & sub["jk_se"].notna() & (sub["jk_se"] > 0)]


def fit_estimator(df, spec):
    lo, hi = spec["range"]
    sub = usable_bins(df, lo, hi)
    a = sub["bin_midpoint"].to_numpy()
    y = sub["full_mean"].to_numpy()
    se = sub["jk_se"].to_numpy()

    X = build_design(a, spec["terms"])
    if spec["intercept"]:
        X = np.column_stack([X, np.ones(len(a))])

    n_bins = len(sub)
    if n_bins <= X.shape[1]:
        return dict(estimator=spec["name"], n_bins=n_bins, point_estimate=np.nan, se=np.nan, dof=0)

    beta, beta_se, dof = wls_fit(X, y, se)

    # b2_FS: b2 = 2 * intercept (intercept is the last column); every other
    # estimator here reports h2 = the slope-on-`a` coefficient (always column 0)
    if spec["name"] == "b2_FS":
        point, point_se = 2 * beta[-1], 2 * beta_se[-1]
    else:
        point, point_se = beta[0], beta_se[0]

    return dict(estimator=spec["name"], n_bins=n_bins, point_estimate=point, se=point_se, dof=dof)


def fit_b2_step(df):
    fs = usable_bins(df, *B2_STEP_FS_RANGE)
    hs = usable_bins(df, *B2_STEP_HS_RANGE)
    if len(fs) <= 2 or len(hs) <= 2:
        return dict(estimator="b2_step", n_bins=len(fs) + len(hs), point_estimate=np.nan, se=np.nan, dof=0)

    X_fs = np.column_stack([fs["bin_midpoint"].to_numpy(), np.ones(len(fs))])
    beta_fs, se_fs, dof_fs = wls_fit(X_fs, fs["full_mean"].to_numpy(), fs["jk_se"].to_numpy())

    X_hs = np.column_stack([hs["bin_midpoint"].to_numpy(), np.ones(len(hs))])
    beta_hs, se_hs, dof_hs = wls_fit(X_hs, hs["full_mean"].to_numpy(), hs["jk_se"].to_numpy())

    intercept_fs, intercept_hs = beta_fs[-1], beta_hs[-1]
    point = 4 * (intercept_fs - intercept_hs)
    # independent fits on disjoint bin ranges -- errors add in quadrature
    point_se = 4 * np.sqrt(se_fs[-1] ** 2 + se_hs[-1] ** 2)
    return dict(estimator="b2_step", n_bins=len(fs) + len(hs), point_estimate=point, se=point_se,
                dof=min(dof_fs, dof_hs))

## Fit every estimator for every combo

In [ ]:
rows = []
for phenotype_name, transform, covariate_set, path in combos:
    df = pd.read_csv(path, sep="\t")
    for spec in ESTIMATORS:
        r = fit_estimator(df, spec)
        r.update(phenotype_name=phenotype_name, transform=transform, covariate_set=covariate_set)
        rows.append(r)
    r = fit_b2_step(df)
    r.update(phenotype_name=phenotype_name, transform=transform, covariate_set=covariate_set)
    rows.append(r)

results = pd.DataFrame(rows)[
    ["phenotype_name", "transform", "covariate_set", "estimator", "n_bins", "point_estimate", "se", "dof"]
]

out_tsv = f"{BUCKET_DIR}/he_estimator_results.tsv"
results.to_csv(out_tsv, sep="\t", index=False)
print(f"Wrote {len(results)} rows to {out_tsv}")
results.head(20)

## Sanity check

`h2_Unrel` here (`a < 0.02`, no intercept) should land close to
`grm_shard_processing.ipynb`'s own dashed slope-line fit for the same
phenotype/transform/covariate_set (`SLOPE_FIT_RANGE = (-0.01, 0.01)`,
same `1/jk_se` weighting convention) -- both are WLS fits over nearly the
same bin range. Large disagreement is a signal something's off in this
notebook's bin filtering, not an expected estimator-definition difference.

In [ ]:
results[results["estimator"] == "h2_Unrel"].sort_values("point_estimate", ascending=False).head(20)

## Forest plot per phenotype/transform

One plot per `(phenotype_name, transform)`, point estimate +/- 1 SE per
estimator, colored by covariate set -- same spirit as the legacy R script's
`p2` comparison plot, adapted for multiple covariate sets side by side.

In [ ]:
PLOTS_DIR = f"{BUCKET_DIR}/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

estimator_order = [e["name"] for e in ESTIMATORS] + ["b2_step"]
cmap = plt.get_cmap("viridis")

saved_plots = []
for (phenotype_name, transform), g in results.groupby(["phenotype_name", "transform"]):
    covsets = sorted(g["covariate_set"].unique())
    covset_color = {cs: cmap(i / max(1, len(covsets) - 1)) for i, cs in enumerate(covsets)}

    fig, ax = plt.subplots(figsize=(6, 4))
    for j, cs in enumerate(covsets):
        d = g[g["covariate_set"] == cs].set_index("estimator").reindex(estimator_order)
        y_pos = np.arange(len(estimator_order)) + j * 0.15
        ax.errorbar(d["point_estimate"], y_pos, xerr=d["se"], fmt="o", ms=4,
                     color=covset_color[cs], label=cs)

    ax.axvline(0, color="grey", lw=0.5)
    ax.set_yticks(np.arange(len(estimator_order)))
    ax.set_yticklabels(estimator_order)
    ax.invert_yaxis()
    ax.set_xlabel("estimate")
    ax.set_title(f"{phenotype_name} ({transform})")
    ax.legend(title="covariate set", fontsize=8)
    plt.tight_layout()

    out_png = f"{PLOTS_DIR}/{phenotype_name}__{transform}__estimators.png"
    fig.savefig(out_png, dpi=150)
    plt.show()
    plt.close(fig)
    saved_plots.append(out_png)

print(f"{len(saved_plots)} plots saved to {PLOTS_DIR}")